# 第一章 初识智能体

## 1.3 动手体验：5 分钟实现第一个智能体

### 1.3.1 准备工作

In [1]:
# 需要的依赖包
#!pip install requests tavily-python openai

In [2]:
# 系统提示词
# 在这里定义了模型需要遵循的行为模式、可以使用的工具
AGENT_SYSTEM_PROMPT = """
你是一个智能旅行助手。你的任务是分析用户的请求，并使用可用工具一步步地解决问题。

# 可用工具:
- `get_weather(city: str)`: 查询指定城市的实时天气。
- `get_attraction(city: str, weather: str)`: 根据城市和天气搜索推荐的旅游景点。

# 行动格式:
你的回答必须严格遵循以下格式。首先是你的思考过程，然后是你要执行的具体行动，每次回复只输出一对 Thought-Action：
Thought: [这里是你的思考过程和下一步计划]
Action: 你决定采取的行动，必须是以下格式之一:
- `function_name(arg_name="arg_value")`:调用一个可用工具。
- `Finish[最终答案]`:当你认为已经获得最终答案时。
- 当你收集到足够的信息，能够回答用户的最终问题时，你必须在Action:字段后使用 Finish[最终答案] 来输出最终答案。

请开始吧！
"""


In [3]:
# 工具 1：查询真实天气
# 我们将使用免费的天气查询服务 wttr.in，它能以 JSON 格式返回指定城市的天气数据。下面是实现该工具的代码：
import requests

def get_weather(city: str) -> str:
    """
    通过调用 wttr.in API 查询真实的天气信息。
    """
    # API端点，我们请求JSON格式的数据
    url = f"https://wttr.in/{city}?format=j1"
    
    try:
        # 发起网络请求
        response = requests.get(url)
        # 检查响应状态码是否为200 (成功)
        response.raise_for_status() 
        # 解析返回的JSON数据
        data = response.json()
        
        print(f"调试: 请求 wttr.in, 参数 city={city}, 返回的原始结果: {data}")  # 调试输出，查看API返回的数据结构
        
        # 提取当前天气状况
        current_condition = data['current_condition'][0]
        weather_desc = current_condition['weatherDesc'][0]['value']
        temp_c = current_condition['temp_C']
        
        # 格式化成自然语言返回
        return f"{city}当前天气: {weather_desc}，气温 {temp_c} 摄氏度"
        
    except requests.exceptions.RequestException as e:
        # 处理网络错误
        return f"错误:查询天气时遇到网络问题 - {e}"
    except (KeyError, IndexError) as e:
        # 处理数据解析错误
        return f"错误:解析天气数据失败，可能是城市名称无效 - {e}"


In [ ]:
# 工具 2：搜索并推荐旅游景点
# 定义一个新工具 get_attraction，它会根据城市和天气状况，互联网上搜索合适的景点：
# 我们将使用 Tavily Search API 来实现这个功能。Tavily 是一个强大的搜索工具，可以帮助我们从互联网上获取相关信息，并且它会自动优化搜索结果，提供一个综合性的回答。
# 使用它需要注册并获取一个 API 密钥，我们将它放在环境变量 TAVILY_API_KEY 中。使用 tavily-python 库来调用 API
# 下面是实现 get_attraction 工具的代码：

import os
from tavily import TavilyClient

def get_attraction(city: str, weather: str) -> str:
    """
    根据城市和天气，使用 Tavily Search API 搜索并返回优化后的景点推荐。
    """
    # 1. 从环境变量中读取API密钥
    api_key = os.environ.get("TAVILY_API_KEY")
    if not api_key:
        return "错误:未配置TAVILY_API_KEY环境变量。"

    # 2. 初始化Tavily客户端
    tavily = TavilyClient(api_key=api_key)
    
    # 3. 构造一个精确的查询
    query = f"'{city}' 在'{weather}'天气下最值得去的旅游景点推荐及理由"
    
    try:
        # 4. 调用API，include_answer=True会返回一个综合性的回答
        response = tavily.search(query=query, search_depth="basic", include_answer=True)
        
        print(f"调试: 调用 Tavily Search API, 查询: {query}, 返回的原始结果: {response}")  # 调试输出，查看API返回的数据结构
        
        # 5. Tavily返回的结果已经非常干净，可以直接使用
        # response['answer'] 是一个基于所有搜索结果的总结性回答
        if response.get("answer"):
            return response["answer"]
        
        # 如果没有综合性回答，则格式化原始结果
        formatted_results = []
        for result in response.get("results", []):
            formatted_results.append(f"- {result['title']}: {result['content']}")
        
        if not formatted_results:
            return "抱歉，没有找到相关的旅游景点推荐。"

        return "根据搜索，为您找到以下信息:\n" + "\n".join(formatted_results)

    except Exception as e:
        return f"错误:执行Tavily搜索时出现问题 - {e}"


In [5]:
# 测试一下工具
get_weather("武汉")

调试: 请求 wttr.in, 参数 city=武汉, 返回的原始结果: {'current_condition': [{'FeelsLikeC': '11', 'FeelsLikeF': '51', 'cloudcover': '0', 'humidity': '67', 'localObsDateTime': '2026-03-14 11:28 PM', 'observation_time': '03:28 PM', 'precipInches': '0.0', 'precipMM': '0.0', 'pressure': '1025', 'pressureInches': '30', 'temp_C': '13', 'temp_F': '56', 'uvIndex': '0', 'visibility': '5', 'visibilityMiles': '3', 'weatherCode': '143', 'weatherDesc': [{'value': 'Haze'}], 'weatherIconUrl': [{'value': ''}], 'winddir16Point': 'NNE', 'winddirDegree': '12', 'windspeedKmph': '32', 'windspeedMiles': '20'}], 'nearest_area': [{'areaName': [{'value': 'Wuhan'}], 'country': [{'value': 'China'}], 'latitude': '30.583', 'longitude': '114.267', 'population': '4184206', 'region': [{'value': 'Hubei'}], 'weatherUrl': [{'value': ''}]}], 'request': [{'query': 'Lat 30.60 and Lon 114.30', 'type': 'LatLon'}], 'weather': [{'astronomy': [{'moon_illumination': '26', 'moon_phase': 'Waning Crescent', 'moonrise': '03:34 AM', 'moonset': '01:43

'武汉当前天气: Haze，气温 13 摄氏度'

In [6]:
get_attraction("武汉", "晴")

调试: 调用 Tavily Search API, 查询: '武汉' 在'晴'天气下最值得去的旅游景点推荐及理由, 返回的原始结果: {'query': "'武汉' 在'晴'天气下最值得去的旅游景点推荐及理由", 'response_time': 1.73, 'follow_up_questions': None, 'answer': '晴川阁和黄鹤楼是武汉晴天最值得去的景点。两者位于长江两岸，景色优美。晴川阁历史悠久，建于明朝。', 'images': [], 'results': [{'url': 'https://www.zhihu.com/question/551554412', 'title': '夏天武汉旅游有哪些景点值得一去？', 'content': '汉阳门这边，龟山桥头派出所，桥上以及晴川阁。每个地方拍照看太阳角度 ... 汤面可以试下顶好以及N多别的店；. 如果想一次吃多种小吃可以去三镇', 'score': 0.71305263, 'raw_content': None}, {'url': 'https://travel.qunar.com/youji/6029619', 'title': '意外的假期武汉之旅', 'content': '爬上去可以继续遥望长江大桥，天气好时应该可以看到对岸的黄鹤楼。 很快就逛完了晴川阁，挺喜欢这里，因为安静。下一站去风格独特的古德寺看看。', 'score': 0.62633044, 'raw_content': None}, {'url': 'https://gs.ctrip.com/html5/you/travels/100067/3984956.html', 'title': '武汉楚河汉街周围有什么好玩的景点值得推荐-美食酒店去 ...', 'content': '武汉是湖北省的省会城市，地处江汉平原，地势多为平坦，市内景区资源丰富，推荐游玩时间为3天，因为景点很多，也可以自驾游，相对来说是最为方便快捷的。. 武汉市大多是人文景区如黄鹤楼、欢乐谷、极地海洋世界、木兰草原等，都是游玩的好去处，想看自然风景区可以去神农架宜昌的等地游玩。. 武汉有什么好玩的特色景点，首先必来打卡的景点就是武汉欢乐谷、玛雅世界、海昌极地世界、长江大桥、黄鹤楼，木兰草原，武汉大学的樱花也是颇有名气，每一个景点都是值得一去，下

'晴川阁和黄鹤楼是武汉晴天最值得去的景点。两者位于长江两岸，景色优美。晴川阁历史悠久，建于明朝。'

In [7]:
# 将所有工具函数放入一个字典，方便后续调用 (告诉 LLM 有哪些 Tools 可以调用)
available_tools = {
    "get_weather": get_weather,
    "get_attraction": get_attraction,
}

### 1.3.2 接入大语言模型

In [8]:
# 可以这个封装很薄，基本上就是直接调用了 OpenAI 的接口。加了一些日志
"""
当前，许多 LLM 服务提供商（包括 OpenAI、Azure、以及众多开源模型服务框架如 Ollama、vLLM 等）都遵循了与 OpenAI API 相似的接口规范。
这种标准化为开发者带来了极大的便利。智能体的自主决策能力来源于 LLM。

我们将实现一个通用的客户端 OpenAICompatibleClient，它可以连接到任何兼容 OpenAI 接口规范的 LLM 服务。
"""
from openai import OpenAI

class OpenAICompatibleClient:
    """
    一个用于调用任何兼容 OpenAI 接口的 LLM 服务的客户端。
    """
    def __init__(self, model: str, api_key: str, base_url: str):
        self.model = model
        self.client = OpenAI(api_key=api_key, base_url=base_url)

    def generate(self, prompt: str, system_prompt: str) -> str:
        """调用LLM API来生成回应。"""
        print("正在调用大语言模型...")
        try:
            messages = [
                {'role': 'system', 'content': system_prompt},
                {'role': 'user', 'content': prompt}
            ]
            response = self.client.chat.completions.create(
                model=self.model,
                messages=messages,
                stream=False
            )
            answer = response.choices[0].message.content
            print("大语言模型响应成功。")
            return answer
        except Exception as e:
            print(f"调用LLM API时发生错误: {e}")
            return "错误:调用语言模型服务时出错。"


### 1.3.3 执行行动循环

In [ ]:
import re
import os

# --- 1. 配置LLM客户端 ---
# 请根据您使用的服务，将这里替换成对应的凭证和地址

# 使用 硅基流动的 API
SILICONFLOW_API_BASE_URL = "https://api.siliconflow.cn/v1"
SILICONFLOW_API_KEY = os.getenv("SILICONFLOW_API_KEY")

API_KEY = SILICONFLOW_API_KEY
BASE_URL = SILICONFLOW_API_BASE_URL
MODEL_ID = "Qwen/Qwen3-VL-32B-Thinking" 
TAVILY_API_KEY = os.getenv("TAVILY_API_KEY")

llm = OpenAICompatibleClient(
    model=MODEL_ID,
    api_key=API_KEY,
    base_url=BASE_URL,
)

# --- 2. 初始化 ---
user_prompt = "你好，请帮我查询一下今天北京的天气，然后根据天气推荐一个合适的旅游景点。"
prompt_history = [f"用户请求: {user_prompt}"]

print(f"用户输入: {user_prompt}\n" + "="*40)

# --- 3. 运行主循环 ---
for i in range(5): # 设置最大循环次数
    print(f"--- 循环 {i+1} ---\n")
    
    # 3.1. 构建Prompt
    full_prompt = "\n".join(prompt_history)
    
    # 3.2. 调用LLM进行思考
    llm_output = llm.generate(full_prompt, system_prompt=AGENT_SYSTEM_PROMPT)
    # 模型可能会输出多余的Thought-Action，需要截断
    match = re.search(r'(Thought:.*?Action:.*?)(?=\n\s*(?:Thought:|Action:|Observation:)|\Z)', llm_output, re.DOTALL)
    if match:
        truncated = match.group(1).strip()
        if truncated != llm_output.strip():
            llm_output = truncated
            print("已截断多余的 Thought-Action 对")
    print(f"模型输出:\n{llm_output}\n")
    prompt_history.append(llm_output)
    
    # 3.3. 解析并执行行动
    action_match = re.search(r"Action: (.*)", llm_output, re.DOTALL)
    if not action_match:
        observation = "错误: 未能解析到 Action 字段。请确保你的回复严格遵循 'Thought: ... Action: ...' 的格式。"
        observation_str = f"Observation: {observation}"
        print(f"{observation_str}\n" + "="*40)
        prompt_history.append(observation_str)
        continue
    action_str = action_match.group(1).strip()

    if action_str.startswith("Finish"):
        final_answer = re.match(r"Finish\[(.*)\]", action_str).group(1)
        print(f"任务完成，最终答案: {final_answer}")
        break
    
    tool_name = re.search(r"(\w+)\(", action_str).group(1)
    args_str = re.search(r"\((.*)\)", action_str).group(1)
    kwargs = dict(re.findall(r'(\w+)="([^"]*)"', args_str))

    if tool_name in available_tools:
        observation = available_tools[tool_name](**kwargs)
    else:
        observation = f"错误:未定义的工具 '{tool_name}'"

    # 3.4. 记录观察结果
    observation_str = f"Observation: {observation}"
    print(f"{observation_str}\n" + "="*40)
    prompt_history.append(observation_str)

用户输入: 你好，请帮我查询一下今天北京的天气，然后根据天气推荐一个合适的旅游景点。
--- 循环 1 ---

正在调用大语言模型...
大语言模型响应成功。
模型输出:


Thought: 首先需要查询北京的实时天气，以便后续根据天气推荐景点。
Action: get_weather(city="北京")

调试: 请求 wttr.in, 参数 city=北京, 返回的原始结果: {'current_condition': [{'FeelsLikeC': '3', 'FeelsLikeF': '37', 'cloudcover': '0', 'humidity': '70', 'localObsDateTime': '2026-03-14 11:23 PM', 'observation_time': '03:23 PM', 'precipInches': '0.0', 'precipMM': '0.0', 'pressure': '1032', 'pressureInches': '30', 'temp_C': '5', 'temp_F': '42', 'uvIndex': '0', 'visibility': '10', 'visibilityMiles': '6', 'weatherCode': '122', 'weatherDesc': [{'value': 'Overcast'}], 'weatherIconUrl': [{'value': ''}], 'winddir16Point': 'SSW', 'winddirDegree': '206', 'windspeedKmph': '12', 'windspeedMiles': '7'}], 'nearest_area': [{'areaName': [{'value': 'Beijing'}], 'country': [{'value': 'China'}], 'latitude': '39.929', 'longitude': '116.388', 'population': '7480601', 'region': [{'value': 'Beijing'}], 'weatherUrl': [{'value': ''}]}], 'request': [{'query': 'Lat 39.91 a

这个简单的旅行助手案例，集中演示了基于Thought-Action-Observation范式的智能体所具备的四项基本能力：任务分解、工具调用、上下文理解和结果合成。正是通过这个循环的不断迭代，智能体才得以将一个模糊的用户意图，转化为一系列具体、可执行的步骤，并最终达成目标。